In [1]:
!pip install -q llama-index

In [2]:
import sys
from pathlib import Path
import pickle
import time
import logging
import numpy as np
from scipy.sparse import csr_matrix, save_npz
from sklearn.feature_extraction.text import CountVectorizer
from llama_index.core import Document

# Add parent directory to Python path
# Assuming your project root is `/work`, add it to the system path
project_root = Path('/work')
sys.path.append(str(project_root))
from src.utils.data_utils import load_documents

# Custom BM25 vectorizer using sparse representations.
class SparseBM25Vectorizer:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.vectorizer = CountVectorizer(binary=False, lowercase=True, token_pattern=r'\b\w+\b')
        self.doc_len = None
        self.avgdl = None
        self.idf = None
        self.doc_freq = None

    def fit(self, documents):
        texts = [doc.text if isinstance(doc, Document) else doc for doc in documents]
        X = self.vectorizer.fit_transform(texts)
        self.doc_len = X.sum(axis=1).A1
        self.avgdl = self.doc_len.mean()
        n_docs = X.shape[0]
        self.doc_freq = X.sum(axis=0).A1
        self.idf = np.log((n_docs - self.doc_freq + 0.5) / (self.doc_freq + 0.5) + 1.0)
        return self

    def transform(self, documents):
        if isinstance(documents, str):
            documents = [documents]
        texts = [doc.text if isinstance(doc, Document) else doc for doc in documents]
        X = self.vectorizer.transform(texts)
        return self._bm25_sparse(X)

    def _bm25_sparse(self, X):
        rows, cols = X.nonzero()
        data = X.data

        doc_len = self.doc_len[rows]
        idf = self.idf[cols]

        numerator = data * (self.k1 + 1)
        denominator = data + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)

        scores = idf * numerator / denominator
        return csr_matrix((scores, (rows, cols)), shape=X.shape)

# Function to save index components.
def save_index(index_path: Path, vectorizer: SparseBM25Vectorizer, embeddings, documents):
    index_path.mkdir(parents=True, exist_ok=True)
    
    with open(index_path / "vectorizer.pkl", "wb") as f:
        pickle.dump(vectorizer, f)
    
    save_npz(index_path / "embeddings.npz", embeddings)
    
    with open(index_path / "documents.pkl", "wb") as f:
        pickle.dump(documents, f)
    
    logging.info(f"Index saved to {index_path}")

# Main function with progress updates and estimated time remaining.
def main(file):
    start = time.time()
    print("Starting indexing process...")

    print("Setting up file paths...")
    documents_path_texto = Path('/work/data/processed/' + file + '_texto.pkl')
    documents_path_resumen = Path('/work/data/processed/' + file + '_resumen.pkl')
    index_path_texto = Path('/work/data/indexes/' + file + '_texto_BM25sparseindex')
    index_path_resumen = Path('/work/data/indexes/' + file + '_resumen_BM25sparseindex')

    print("Loading documents for 'texto'...")
    documents_texto = load_documents(documents_path_texto)
    print("Loading documents for 'resumen'...")
    documents_resumen = load_documents(documents_path_resumen)

    if documents_texto is None or documents_resumen is None:
        logging.error("Failed to load one or both document sets. Exiting.")
        return

    datasets = [
        ("texto", documents_texto, index_path_texto),
        ("resumen", documents_resumen, index_path_resumen)
    ]
    total_steps = len(datasets)

    for i, (name, docs, index_path) in enumerate(datasets, 1):
        print(f"\nProcessing {name} documents ({i}/{total_steps})...")
        step_start = time.time()
        
        print("Fitting BM25 vectorizer...")
        vectorizer = SparseBM25Vectorizer().fit(docs)
        
        print("Transforming documents to BM25 embeddings...")
        embeddings = vectorizer.transform(docs)

        print("Saving index for", name, "documents...")
        save_index(index_path, vectorizer, embeddings, docs)

        step_end = time.time()
        step_elapsed = step_end - step_start
        remaining_steps = total_steps - i
        estimated_remaining = step_elapsed * remaining_steps if remaining_steps > 0 else 0
        minutes, seconds = divmod(estimated_remaining, 60)
        print(f"Completed processing {name} documents in {step_elapsed:.2f} seconds. "
              f"Estimated time remaining: {int(minutes)} minutes, {int(seconds)} seconds.")

    end = time.time()
    total_time = end - start
    print(f"\nTotal execution time: {total_time:.2f} seconds")


In [3]:
main('OEB')

Starting indexing process...
Setting up file paths...
Loading documents for 'texto'...
Loading documents for 'resumen'...

Processing texto documents (1/2)...
Fitting BM25 vectorizer...
Transforming documents to BM25 embeddings...
Saving index for texto documents...
Completed processing texto documents in 3.96 seconds. Estimated time remaining: 0 minutes, 3 seconds.

Processing resumen documents (2/2)...
Fitting BM25 vectorizer...
Transforming documents to BM25 embeddings...
Saving index for resumen documents...
Completed processing resumen documents in 2.94 seconds. Estimated time remaining: 0 minutes, 0 seconds.

Total execution time: 10.84 seconds
